# WORK IN PROGRESS
Atte. Raúl

In [1]:
import os

import tensorflow as tf
import numpy as np

from sklearn.metrics import classification_report

from TheModel import build

### 1. Carga de los modelos locales

In [ ]:
loaded_local_models = [tf.keras.models.load_model(os.path.join(root, file)) for root, dirs, files in os.walk("./") for file in files if file.endswith('.keras')]

for i in range(len(loaded_local_models)-1):
    assert loaded_local_models[i].summary() == loaded_local_models[i+1].summary(), "Models have different architectures"

### 2. Modelo global (promedio de pesos)

El modelo global se obtiene promediando capa por capa los pesos de los modelos locales.

In [ ]:
train, test = tf.keras.datasets.mnist.load_data()

x_train, x_test = np.expand_dims(train[0] / 255.0, -1), np.expand_dims(test[0] / 255.0, -1)
y_train, y_test = train[1], test[1]

In [ ]:
local_weights = [x.get_weights() for x in loaded_local_models]
averaged_weights = [np.mean(np.array(weights), axis=0) for weights in zip(*local_weights)]

global_model = build.build_it()
global_model.set_weights(averaged_weights)

y_pred = global_model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
print(classification_report(y_test, y_pred_classes))

global_model.save('global_model.keras')